## Imports

In [3]:
import os
import pandas as pd
import csv
import json
import glob
import logging

In [28]:
def get_qualisys_metadata(filename: str) -> dict:
    """
    Reads qualisys export file in .tsv format along with (custom) json files, parses metadata.
    Exported file has to include tsv-header (qualisys export setting).

    Args:
        filename (str): path to the qualisys export file in .tsv format

    Returns:
        metadata in dict format
    """

    qualisys_metadata = {}
    with open(filename) as fd:
        # read file with csv reader. no need for pandas for just a few lines
        reader = csv.reader(fd, delimiter="\t", quotechar='"')

        for ind, row in enumerate(reader):
            # only the first 9 rows of the whole file contain the tsv header metadata
            if ind <= 8:
                if ind < 6:
                    qualisys_metadata[row[0]] = float(row[1])
                # this row has 2 pieces of info; timestamp of the recording from qualisys (this cant be used for sync), timestamp from the start of host system
                elif ind == 7:
                    qualisys_metadata[row[0] + "_QUALISYS"] = row[1]
                    qualisys_metadata[row[0] + "_FROM_SYSTEM_START"] = row[2]
                else:
                    qualisys_metadata[row[0]] = row[1]

    return qualisys_metadata


def read_improper_json_file(filename: str) -> dict:
    """
    Reads a json file that has been improperly formatted (e.g. single quotes instead of double quotes).
    Args:
        filename (str): path to the json file

    Returns:
        json_data (dict): data with formatting corrected.
    """
    path = os.path.join(".", filename)
    if os.path.isfile(path):
        with open(path, "r") as f:
            content = f.read()
            json_data = json.loads(content.replace("'", '"'))

    return json_data


def get_json_metadata() -> dict:
    """
    Reads two json metadata files (w predefined names and path) from the start and stop of a recording session.
    Corrects the formatting (' to ") and checks for consistency between the two files.
    Puts the unified metadata into a single dictionary.

    Args:
        None

    Returns:
        unified_json_data (dict): a dictionary containing the unified metadata from both json files.
    """

    # read in both metadata json files
    data_start = read_improper_json_file("data_start.json")
    data_stop = read_improper_json_file("data_stop.json")

    unified_json_data = {}

    # check if both have the same field names (keys)
    if data_start.keys() == data_stop.keys():

        # check if the uuid is the same in both
        if data_start["uuid"] == data_stop["uuid"]:
            unified_json_data["uuid"] = data_start["uuid"]

        else:
            msg = f'uuid different for {data_start["filename"]}'
            logging.error(msg)

        # check if the filename is the same in both

        if data_start["filename"] == data_stop["filename"]:
            unified_json_data["filename"] = data_start["filename"]

        else:

            msg = f'filename different for {data_start["filename"]}'
            logging.error(msg)

        # check if the timestamps are different (they should be)
        if data_start["timestamp"] != data_stop["timestamp"]:
            unified_json_data["timestamp_start"] = data_start["timestamp"]
            unified_json_data["timestamp_stop"] = data_stop["timestamp"]

        else:
            msg = f'timestamps are identical for  {data_start["filename"]}'
            logging.error(msg)

        # check if the timestamps for qualisys ARE different (they should be)
        if data_start["timestamp_qualisys"] != data_stop["timestamp_qualisys"]:
            unified_json_data["timestamp_qualisys_start"] = data_start["timestamp_qualisys"]
            unified_json_data["timestamp_qualisys_stop"] = data_stop["timestamp_qualisys"]

        else:
            msg = f'timestamp_qualisys are identical for {data_start["filename"]}'
            logging.error(msg)
    else:
        msg = f'keys of json files are different: {set(data_start.keys()).difference(data_stop.keys())} for {data_start["filename"]}'
        logging.error(msg)

    return unified_json_data


# get_qualisys_metadata("./tsv_exp_header_colhead_time.tsv")

In [ ]:
# logging.basicConfig(
#     filename="./process_qualisys_data.log",
#     encoding="utf-8",
#     filemode="a",
#     format="{asctime} - {levelname} - {message}",
#     style="{",
#     datefmt="%Y-%m-%d %H:%M",
# )

# TODO: add logging to file

# read in and process json metadata files
json_metadata = get_json_metadata()

# read in and process metadata (header) from qualisys export file
qualisys_metadata = get_qualisys_metadata("./tsv_exp_header_colhead_time.tsv")

# combine the two dictionaries 
# note: this will overwrite data with the same keys, they have to be unique
metadata = qualisys_metadata | json_metadata

# write result to a json file
with open("./all_metadata.json", "w") as f:
    json.dump(metadata, f)

In [ ]:
# metadata = pd.read_csv('./tsv_exp_header_colhead_time_skeleton.tsv',sep='\t',nrows=10)
# metadata.head()

# df = pd.read_csv('./tsv_exp_header_colhead_skeleton.tsv',sep='\t',skiprows=10)
# pd.read_csv("mydata.csv", skiprows=10000 nrows=10000)

In [ ]:
# write json data to file
with open('data.json', 'w') as f:
    json.dump(dic, f)

set()